**Run note:** execute this notebook's first setup/code cell before any later cells. Each notebook is designed to run independently and re-detect the dataset path on its own.

# 01 â€” Problem Scope & Design Decisions

Defines the exact task, metrics, dataset choice, and architecture strategy before any code is written.

## 1.1 Task Definition

**Task**: Binary multimodal classification â€” hateful (1) vs non-hateful (0) meme detection.

**Why binary?**
- Meta Hateful Memes Challenge uses binary labels (0/1).
- Multi-class (offensive, sarcasm, implicit hate) requires additional annotation not present in the base dataset.
- A strong binary baseline is prerequisite to multi-class extension.

**Scope boundary** (what this project detects):
- Overt hate speech in memes (slurs, dehumanization, calls for violence)
- Hate that requires **both** image and text modalities to identify (neither alone is sufficient)
- NOT: general offensive content, sarcasm without hate, clickbait

## 1.2 Dataset

| Property | Value |
|---|---|
| Name | Meta AI Hateful Memes Challenge |
| Train size | ~8,500 samples |
| Dev size | ~500 samples |
| Test size | ~1,000 samples |
| Labels | 0 = non-hateful, 1 = hateful |
| Image format | PNG |
| Text source | OCR text embedded in the JSONL (human-cleaned) |
| Challenge URL | https://ai.facebook.com/tools/hatefulmemes/ |

**Confounders in dataset** (pairs designed to test multimodal understanding):
- Benign image + hateful text â†’ hateful
- Hateful image + benign text â†’ non-hateful
- Same image, different text â†’ different labels

## 1.3 Success Metrics

Primary metric: **AUROC** (area under ROC curve) â€” matches the original challenge metric.

| Metric | Description | Target |
|---|---|---|
| AUROC | Ranking quality, threshold-independent | â‰¥ 0.75 |
| Macro F1 | Average of F1 for both classes | â‰¥ 0.70 |
| F1 (hateful) | Recall on hateful class matters most | â‰¥ 0.65 |
| Accuracy | Overall correctness | â‰¥ 0.72 |

**Why AUROC is primary**: Threshold-independent; critical for content moderation where you tune the threshold per deployment context (low false negatives for safety vs low false positives for user experience).

## 1.4 Architecture Decisions

| Component | Choice | Reason |
|---|---|---|
| Text encoder | CLIP text encoder (ViT-B/32) | Aligned with image encoder; no separate text model needed |
| Image encoder | CLIP vision encoder (ViT-B/32) | Strong zero-shot visual features; pretrained on 400M image-text pairs |
| Fusion | Cross-attention (bidirectional) | Forces model to learn inter-modal dependencies, not just concatenate |
| Classifier | MLP (1024â†’256â†’128â†’2) | Lightweight, interpretable, fast to train |
| Encoder weights | Frozen | Limited GPU budget on Kaggle; full fine-tuning risks forgetting |
| Class imbalance | Weighted Random Sampler + Focal Loss | WRS ensures balanced batches; Focal Loss down-weights easy negatives |

**Baseline ladder** (to prove multimodal fusion adds value):
1. TF-IDF + Logistic Regression (text only)
2. CLIP text encoder + linear head (text only)
3. CLIP image encoder + linear head (image only)
4. Concatenated CLIP embeddings + MLP (simple fusion)
5. Cross-attention fusion (our main model)

## 1.5 Responsible AI Components

| Component | Implementation |
|---|---|
| Explainability | GradCAM (image regions) + token attribution (text) |
| Counterfactuals | Minimal edits that flip prediction (text token swap + image patch occlusion) |
| Fairness audit | Per-demographic-group performance evaluation |
| Calibration | Reliability diagram â€” model confidence should match actual accuracy |
| Human-in-the-loop | Three-tier routing (auto-approve / human-review / auto-remove) |
| Robustness | OCR noise injection, paraphrase testing |

## 1.6 Deliverables

1. **Research prototype**: All ablations trained and evaluated
2. **Explainability layer**: GradCAM + attention weights
3. **Counterfactual module**: Automated minimal-flip generation
4. **Web demo**: Gradio 4-tab interface (predict / explain / counterfactual / flag)
5. **Backend API**: FastAPI server with prediction endpoint
6. **Audit log**: JSONL of all HITL annotations
7. **Error analysis**: Grouped failure case report

In [1]:
# Quick sanity check: load one sample
import json
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").is_dir()
JSONL_CANDIDATES = {
    "train": ["train.jsonl"],
    "dev": ["dev.jsonl", "dev_seen.jsonl", "dev_unseen.jsonl"],
    "test": ["test.jsonl", "test_seen.jsonl", "test_unseen.jsonl"],
}
IMAGE_DIR_CANDIDATES = ("img", "images")


def _has_image_dir(path: Path) -> bool:
    return any((path / name).is_dir() for name in IMAGE_DIR_CANDIDATES)


def _has_any_jsonl(path: Path, names) -> bool:
    return any((path / name).is_file() for name in names)


def _looks_like_dataset_root(path: Path) -> bool:
    return path.is_dir() and _has_image_dir(path) and _has_any_jsonl(path, JSONL_CANDIDATES["train"])


def detect_data_dir():
    for env_name in ("KAGGLE_DATA_DIR", "META_HATEFUL_MEME_DATA_DIR"):
        env_dir = os.environ.get(env_name, "").strip()
        if env_dir and _looks_like_dataset_root(Path(env_dir)):
            return Path(env_dir), f"env:{env_name}"

    kaggle_input = Path("/kaggle/input")
    default_candidate = kaggle_input / "meta-hateful-meme-detection" / "data"
    if _looks_like_dataset_root(default_candidate):
        return default_candidate, "default:/kaggle/input/meta-hateful-meme-detection/data"

    if ON_KAGGLE:
        for train_jsonl in sorted(kaggle_input.rglob("train.jsonl")):
            candidate = train_jsonl.parent
            if _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

        for candidate in sorted(kaggle_input.rglob("*")):
            if candidate.is_dir() and _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

    for candidate in (Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd(), Path.cwd().parent):
        if _looks_like_dataset_root(candidate):
            return candidate, f"local:{candidate}"

    return None, "not-found"


def resolve_split(base_dir, names):
    base_dir = Path(base_dir)
    for name in names:
        path = base_dir / name
        if path.is_file():
            return path
    for name in names:
        matches = sorted(base_dir.rglob(name))
        if matches:
            return matches[0]
    return None


DATA_DIR, data_source = detect_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Set KAGGLE_DATA_DIR or META_HATEFUL_MEME_DATA_DIR to the folder containing train.jsonl and img/."
    )

IMG_DIR = next((DATA_DIR / name for name in IMAGE_DIR_CANDIDATES if (DATA_DIR / name).is_dir()), None)
TRAIN_PATH = resolve_split(DATA_DIR, JSONL_CANDIDATES["train"])
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if TRAIN_PATH is None:
    raise FileNotFoundError(f"Train split not found under {DATA_DIR}")

DATA_DIR = str(DATA_DIR)
IMG_DIR = str(IMG_DIR) if IMG_DIR is not None else None
TRAIN_PATH = str(TRAIN_PATH)
OUTPUT_DIR = str(OUTPUT_DIR)

print(f"Using DATA_DIR : {DATA_DIR}")
print(f"Using IMG_DIR  : {IMG_DIR}")
print(f"Using source   : {data_source}")
print(f"Output dir     : {OUTPUT_DIR}")

with open(TRAIN_PATH, encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("Sample record:")
for k, v in sample.items():
    print(f"  {k}: {v}")
print("\nFields match expected schema:", set(sample.keys()) >= {"id", "img", "label", "text"})

Using DATA_DIR : /kaggle/input/datasets/muddybuddy/meta-hateful-meme-detection/data
Using IMG_DIR  : /kaggle/input/datasets/muddybuddy/meta-hateful-meme-detection/data/img
Using source   : auto:/kaggle/input/datasets/muddybuddy/meta-hateful-meme-detection/data
Output dir     : /kaggle/working
Sample record:
  id: 42953
  img: img/42953.png
  label: 0
  text: its their character not their color that matters

Fields match expected schema: True
